# Audit and Oversight

Autonomous agents make decisions continuously, but accountability doesn't disappear because a machine made the call. Every agent action still carries organizational weight - which means every decision needs to be traceable: what was requested, what the agent chose to do, why, and what happened afterward.

The challenge isn't logging itself - that is straightforward. The real challenge is making logged data *actionable*. Raw logs accumulate; what human overseers need is compliance checking that fires before a problematic action, reaches a customer, a risk assessment that routes borderline cases to a reviewer rather than silently, auto-approving them, and an audit trail structured enough to support a post-incident investigation.

This notebook implements that oversight loop with LangGraph: an agent decides how to handle a customer service request, a compliance node checks the decision against policy rules, a conditional edge routes high-risk actions to a human reviewer, and every node writes a structured entry to an audit trail that grows across the full workflow.

In [1]:
import os
import uuid
from datetime import datetime
from typing import TypedDict, Literal, Optional, List, Dict, Any
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

### Initialize the language model

In [2]:
# Low temperature — consistent decisions make compliance checking and audit trails
# more meaningful; we want the same input to produce the same decision reliably
llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)

## State design
The state carries everything the oversight system needs across the full workflow: what was requested, what the agent decided, the compliance and risk assessment, any human review outcome, and the cumulative audit trail.

Two design decisions matter here. First, `compliance_violations` is a list of descriptive strings rather than a boolean flag - a list lets the risk node count violations and calibrate severity, and it makes the audit trail self-explanatory without requiring a lookup table. Second, `audit_trail` is a list of dicts that each node extends; by the time the graph reaches `END`, the trail is a complete ordered record of every event in the workflow - agent decision, compliance result, human review, and final outcome.

In [3]:
class AuditState(TypedDict):
    # Input: what was requested and any domain context the compliance rules check against (e.g. refund_amount, account_age_days, action_type)
    request: str
    context: Optional[Dict[str, Any]]

    # Agent decision: populated by execute_action
    action_id: Optional[str]
    action_description: Optional[str]
    agent_reasoning: Optional[str]
    confidence: Optional[float]           # LLM self-assessment; drives risk level

    # Compliance and routing: populated by assess_compliance
    compliance_violations: List[str]      # empty list = fully compliant
    risk_level: Optional[Literal['low', 'medium', 'high']]
    requires_review: bool                 # True routes to the human_review node

    # Human review outcome: only set when requires_review is True
    review_decision: Optional[Literal['approve', 'reject', 'modify']]
    review_notes: Optional[str]

    # Chronological event log: every node appends one entry; never mutated in-place
    audit_trail: List[Dict]

## Executing the agent action
The first node asks the LLM to decide what to do with the customer request and, crucially, to expose its reasoning and self-assessed confidence as structured output. Confidence matters because it flows directly into the compliance node - an agent uncertain about its own decision is a higher-risk signal than one that is confident, even if no explicit policy rule was violated.

The prompt asks for three labelled fields (`ACTION`, `REASONING`, `CONFIDENCE`) so the parser can extract them line by line without any fragile regex. If confidence can't be parsed as a float — unlikely but possible - the fallback is `0.5`, which sits below the `0.7` threshold that triggers medium risk. That means parse failures route toward review rather than silently auto-approving an ambiguous decision.

In [4]:
def execute_action(state: AuditState) -> dict:
    request = state['request']
    context = state.get('context') or {}

    # Flatten context dict into readable lines so the LLM sees the domain data
    # (amounts, account age, action type) rather than raw JSON
    ctx_lines = [f'  {k}: {v}' for k, v in context.items()]
    ctx_text = '\n'.join(ctx_lines) if ctx_lines else '  (no additional context)'

    prompt_lines = [
        'You are a customer service agent deciding how to handle a request.',
        '',
        f'REQUEST: {request}',
        '',
        'Context:',
        ctx_text,
        '',
        'Decide what action to take. Reply in exactly this format:',
        'ACTION: <what you will do>',
        'REASONING: <why>',
        'CONFIDENCE: <0.0 to 1.0>',
    ]
    prompt = '\n'.join(prompt_lines)

    response = llm.invoke(prompt)
    response_text = response.content.strip()

    # Parse the three labelled fields; default confidence to 0.5 so that
    # any parse failure routes toward review rather than auto-approval
    action_desc = ''
    reasoning = ''
    confidence = 0.5

    for line in response_text.split('\n'):
        if line.startswith('ACTION:'):
            action_desc = line.split(':', 1)[1].strip()
        elif line.startswith('REASONING:'):
            reasoning = line.split(':', 1)[1].strip()
        elif line.startswith('CONFIDENCE:'):
            raw = line.split(':', 1)[1].strip()
            try:
                confidence = float(raw)
            except ValueError:
                pass  # keep safe default 0.5

    action_id = 'ACT-' + uuid.uuid4().hex[:8].upper()

    # First audit trail entry: captures the original decision and confidence
    # so investigators have a complete record of what the agent chose to do
    trail_entry = {
        'event': 'action_decided',
        'timestamp': datetime.now().isoformat(),
        'action_id': action_id,
        'action': action_desc,
        'confidence': confidence,
    }

    print(f'Action ID : {action_id}')
    print(f'Decision  : {action_desc}')
    print(f'Confidence: {confidence:.2f}')

    return {
        'action_id': action_id,
        'action_description': action_desc,
        'agent_reasoning': reasoning,
        'confidence': confidence,
        'audit_trail': state.get('audit_trail', []) + [trail_entry],
    }

- `ctx_text` flattens `context` into a prompt-readable block - the LLM sees `refund_amount: 350` rather than a JSON blob.
- `line.split(':', 1)` uses `maxsplit=1` so a colon inside a value (e.g. a URL) doesn't break parsing.
- `confidence = 0.5` is the safe default: `0.5 < 0.7` triggers medium risk in the next node.
- `audit_trail` is extended with `state.get('audit_trail', []) + [trail_entry]` - never mutated in-place.

## Assessing compliance and risk
With the decision made, the next node checks it against policy rules and assigns a risk level. All three rules are deterministic - no LLM involved, because policy checking needs to be consistent and auditable rather than probabilistic. The risk level becomes the routing signal: `low` auto-approves, `medium` or `high` routes to a human reviewer.

In [5]:
def assess_compliance(state: AuditState) -> dict:
    context = state.get('context') or {}
    confidence = state.get('confidence', 1.0)
    violations = []

    # Rule 1: refunds above $200 require manager sign-off
    refund_amount = context.get('refund_amount', 0)
    if refund_amount > 200:
        violations.append(f'refund_exceeds_threshold (${refund_amount} > $200)')

    # Rule 2: accounts under 30 days old are restricted from refunds and account changes
    account_age = context.get('account_age_days', 999)
    action_type = context.get('action_type', '')
    if account_age < 30 and action_type in ('refund', 'account_change'):
        violations.append(f'new_account_restricted (age={account_age}d, action={action_type})')

    # Rule 3: account suspension always requires human sign-off regardless of other factors
    if action_type == 'suspension':
        violations.append('suspension_requires_review')

    # Risk level: violation count drives severity; low confidence tips a borderline case upward
    n = len(violations)
    if n >= 2:
        risk_level = 'high'
    elif n == 1 or confidence < 0.7:
        risk_level = 'medium'
    else:
        risk_level = 'low'

    requires_review = risk_level in ('medium', 'high')

    trail_entry = {
        'event': 'compliance_assessed',
        'timestamp': datetime.now().isoformat(),
        'violations': violations,
        'risk_level': risk_level,
        'requires_review': requires_review,
    }

    status = 'REQUIRES REVIEW' if requires_review else 'AUTO-APPROVED'
    print(f'Violations : {violations if violations else "none"}')
    print(f'Risk level : {risk_level.upper()}')
    print(f'Routing    : {status}')

    return {
        'compliance_violations': violations,
        'risk_level': risk_level,
        'requires_review': requires_review,
        'audit_trail': state.get('audit_trail', []) + [trail_entry],
    }

## Human review and finalisation
Three short functions handle the end of the workflow. `route_for_review` reads `requires_review` and returns the next node name for the conditional edge. `human_review` is where execution pauses for real human input - in production this would be a webhook or API callback; here we use `input()` to simulate the reviewer. `finalize` writes the last audit trail entry and determines the ultimate outcome.

The `human_review` function presents the full context - action, reasoning, violations — so the reviewer can make an informed decision without looking anything up. Any input other than an explicit `approve` or `modify` is treated as a rejection: ambiguity must never result in an action proceeding.

In [6]:
def route_for_review(state: AuditState) -> str:
    # Low-risk actions skip human review entirely; routing logic stays one line
    # because any complexity here belongs in assess_compliance, not the router
    return 'review' if state['requires_review'] else 'finalize'


def human_review(state: AuditState) -> dict:
    # In production: pause here, notify a reviewer, and resume on webhook or API callback.
    # LangGraph's interrupt() + MemorySaver pattern handles this for long-lived workflows.
    print('\n--- Human Review Required ---')
    print(f'Action ID : {state.get("action_id")}')
    print(f'Risk      : {state.get("risk_level", "").upper()}')
    print(f'Action    : {state.get("action_description")}')
    print(f'Reasoning : {state.get("agent_reasoning")}')
    print(f'Violations: {state.get("compliance_violations", [])}')

    raw = input('\nDecision (approve / reject / modify): ').strip().lower()
    notes = input('Notes for audit log (press Enter to skip): ').strip()

    # Map input to decision; anything other than explicit approve/modify is a rejection
    if raw in ('approve', 'a'):
        decision = 'approve'
    elif raw in ('modify', 'm'):
        decision = 'modify'
    else:
        decision = 'reject'

    print(f'Decision  : {decision.upper()}')

    trail_entry = {
        'event': 'human_review_completed',
        'timestamp': datetime.now().isoformat(),
        'decision': decision,
        'notes': notes or 'none',
    }

    return {
        'review_decision': decision,
        'review_notes': notes or 'No notes provided',
        'audit_trail': state.get('audit_trail', []) + [trail_entry],
    }


def finalize(state: AuditState) -> dict:
    # Translate review_decision (or its absence) into a plain outcome string
    # that makes aggregate audit queries easy: group by outcome to spot patterns
    review_decision = state.get('review_decision')
    if review_decision == 'reject':
        outcome = 'blocked'
    elif review_decision in ('approve', 'modify'):
        outcome = 'approved_after_review'
    else:
        # No review node ran — action was auto-approved due to low risk
        outcome = 'auto_approved'

    trail_entry = {
        'event': 'action_finalized',
        'timestamp': datetime.now().isoformat(),
        'action_id': state.get('action_id'),
        'outcome': outcome,
    }

    print(f'\nOutcome: {outcome.upper()}')

    return {
        'audit_trail': state.get('audit_trail', []) + [trail_entry],
    }

- `route_for_review` stays one line — routing logic should be obvious at a glance; any branching complexity belongs in `assess_compliance`, not here.
- `human_review` models the production pause point: in a real system this function would write the pending action to a database and the graph would suspend until a reviewer responds via webhook or API call.
- `finalize` uses `review_decision` as its signal rather than `requires_review` so the outcome accurately reflects what actually happened, not just what was planned.
- The three outcome strings (`auto_approved`, `approved_after_review`, `blocked`) are designed to be filterable in aggregate queries: group by outcome to spot patterns across thousands of actions.

## Assembling the graph
The graph has four nodes connected by one deterministic edge, one conditional edge, and one post-review edge that rejoins at `finalize`. Low-risk actions take the short path (`execute → assess → finalize`); high-risk actions take the long path (`execute → assess → review → finalize`). Both paths produce a complete audit trail.

In [7]:
workflow = StateGraph(AuditState)

workflow.add_node('execute', execute_action)
workflow.add_node('assess', assess_compliance)
workflow.add_node('review', human_review)
workflow.add_node('finalize', finalize)

workflow.set_entry_point('execute')

# execute always flows to assess; every action needs compliance checking
workflow.add_edge('execute', 'assess')

# assess branches: low risk → finalize (auto-approved), medium/high → review
workflow.add_conditional_edges(
    'assess',
    route_for_review,
    {'review': 'review', 'finalize': 'finalize'},
)

# after human review (approve or reject) always proceed to finalize
workflow.add_edge('review', 'finalize')
workflow.add_edge('finalize', END)

audit_graph = workflow.compile()

## Testing two paths through the graph
Two requests exercise the two paths. The first is a policy question - no compliance rule fires, confidence will be high, and the action should auto-approve through the short path. The second is a large refund on a new account - two rules fire simultaneously, risk is high, and the action routes to `human_review` before finalisation.

**Test 2 requires human input.** When prompted, enter `approve`, `reject`, or `modify` and an optional note.

In [8]:
# --- Test 1: information request, expected outcome: auto_approved ---
print('=== Test 1: Low-Risk Information Request ===\n')

state_1 = {
    'request': 'Can you explain what your return policy is?',
    'context': {'action_type': 'information'},
    'compliance_violations': [],
    'requires_review': False,
    'audit_trail': [],
}

result_1 = audit_graph.invoke(state_1)

=== Test 1: Low-Risk Information Request ===

Action ID : ACT-C24AFEDA
Decision  : Provide a detailed explanation of the return policy.
Confidence: 1.00
Violations : none
Risk level : LOW
Routing    : AUTO-APPROVED

Outcome: AUTO_APPROVED


In [9]:
# --- Test 2: large refund on a new account, expected outcome: approved_after_review ---
# Two rules fire: refund_exceeds_threshold and new_account_restricted → risk = high
print('\n=== Test 2: High-Risk Refund on New Account ===\n')

state_2 = {
    'request': 'I want a full refund of $350. I am unhappy with my recent order.',
    'context': {
        'action_type': 'refund',
        'refund_amount': 350,
        'account_age_days': 12,
    },
    'compliance_violations': [],
    'requires_review': False,
    'audit_trail': [],
}

result_2 = audit_graph.invoke(state_2)


=== Test 2: High-Risk Refund on New Account ===

Action ID : ACT-1118BA8F
Decision  : Approve the full refund of $350.
Confidence: 0.90
Violations : ['refund_exceeds_threshold ($350 > $200)', 'new_account_restricted (age=12d, action=refund)']
Risk level : HIGH
Routing    : REQUIRES REVIEW

--- Human Review Required ---
Action ID : ACT-1118BA8F
Risk      : HIGH
Action    : Approve the full refund of $350.
Reasoning : The customer has requested a full refund and has only been a customer for 12 days, which suggests they may not have had sufficient time to fully experience the product or service. Additionally, customer satisfaction is important, and addressing their unhappiness promptly can help maintain a positive relationship.
Violations: ['refund_exceeds_threshold ($350 > $200)', 'new_account_restricted (age=12d, action=refund)']



Decision (approve / reject / modify):  Modify
Notes for audit log (press Enter to skip):  Can refund just 50%


Decision  : MODIFY

Outcome: APPROVED_AFTER_REVIEW


In [10]:
# Print the complete audit trail for both results
# Each entry shows the event name, timestamp, and all associated fields
def print_audit_trail(result: dict, label: str) -> None:
    trail = result.get('audit_trail', [])
    print(f'\n=== Audit Trail: {label} ===')
    for i, entry in enumerate(trail, 1):
        # Trim ISO timestamp to HH:MM:SS to keep output compact
        ts = entry['timestamp'][11:19]
        event = entry['event']
        print(f'  [{i}] {ts}  {event}')
        for k, v in entry.items():
            if k not in ('event', 'timestamp'):
                print(f'       {k}: {v}')
    print()


print_audit_trail(result_1, 'Low-Risk Request')
print_audit_trail(result_2, 'High-Risk Refund')


=== Audit Trail: Low-Risk Request ===
  [1] 14:34:59  action_decided
       action_id: ACT-C24AFEDA
       action: Provide a detailed explanation of the return policy.
       confidence: 1.0
  [2] 14:34:59  compliance_assessed
       violations: []
       risk_level: low
       requires_review: False
  [3] 14:34:59  action_finalized
       action_id: ACT-C24AFEDA
       outcome: auto_approved


=== Audit Trail: High-Risk Refund ===
  [1] 14:35:18  action_decided
       action_id: ACT-1118BA8F
       action: Approve the full refund of $350.
       confidence: 0.9
  [2] 14:35:18  compliance_assessed
       violations: ['refund_exceeds_threshold ($350 > $200)', 'new_account_restricted (age=12d, action=refund)']
       risk_level: high
       requires_review: True
  [3] 14:36:11  human_review_completed
       decision: modify
       notes: Can refund just 50%
  [4] 14:36:11  action_finalized
       action_id: ACT-1118BA8F
       outcome: approved_after_review



The workflow built in this notebook shows how audit and oversight integrate cleanly into an agent pipeline: every action is logged, every risky action is routed to a human, and every decision — including the human's reasoning - is captured in a structured audit trail.

When to use audit and oversight patterns:
- Agents that act on behalf of users in regulated domains (finance, healthcare, legal) where every decision must be traceable.
- Pipelines where some actions require human authority - fee waivers, policy exceptions, account suspensions.
- Any system where a post-incident investigation might need to reconstruct exactly what happened and why.

Design principles that matter in production:
- Keep compliance rules deterministic. LLM-based compliance checking is non-deterministic and hard to audit; fixed rule functions are transparent, testable, and tunable without touching a prompt.
- Use violation strings, not boolean flags. A list of descriptive strings makes the audit trail self-explanatory and allows the risk node to calibrate severity by violation count.
- Route on actual human decisions, not automated surrogates. A `human_review` node that makes its own decisions based on violation type is not human-in-the-loop — it is more automation with a misleading name.
- Write audit trails to an external append-only store in production. State stored in the workflow engine does not persist beyond the run lifecycle and is not available for compliance reporting or cross-run analysis.
- Use descriptive outcome strings (`auto_approved`, `approved_after_review`, `blocked`) rather than booleans. Aggregate queries like "what fraction of high-risk actions were blocked last month" require groupable values, not flags.